# Teste da calculadora de títulos públicos (`titulospub`)

Este notebook demonstra um **fluxo completo de uso do pacote `titulospub`**, o mesmo núcleo de cálculo utilizado pela API FastAPI e pela aplicação Dash.

Ele cria exemplos de títulos LTN, LFT, NTNB e NTNF, define posições (quantidade/financeiro) e exibe resultados principais (PU, taxa, DV01 etc.), além de um exemplo de **equivalência entre títulos**.

## Pré-requisitos

1. Estar na raiz do repositório `calculadora_titulos_publicos`.
2. Ter instalado as dependências e o pacote em modo desenvolvimento:

```bash
pip install -r requirements.txt
pip install -e .
```

Depois disso, `import titulospub` deve funcionar normalmente neste notebook.

In [3]:
from datetime import datetime

import pandas as pd

from titulospub import LTN, LFT, NTNB, NTNF, equivalencia, VariaveisMercado

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [4]:
var = VariaveisMercado(
)

var.atualizar_tudo()

Atualizando variáveis de mercado...
Buscando feriados via scraping...
Calculando IPCA dict...
Buscando CDI...
Realizando scraping ANBIMA...
[OK] Cache salvo para todos os títulos ANBIMA.
Realizando scraping BMF...
[INFO] Encontrados 4 arquivos com prefixo Interest_Rate_SettlementPriceFile_Futures_20260519_
[INFO] Arquivo mais recente selecionado: Interest_Rate_SettlementPriceFile_Futures_20260519_3.csv
[INFO] Data de modificação: Tue May 19 18:53:21 2026
[OK] Cache salvo para todos os contrados de DI e DAP.
Realizando scraping VNA_LFT...
[OK] Cache salvo para VNA_LFT.
[OK] Atualização concluída.


In [6]:
var._feriados

[Timestamp('2001-01-01 00:00:00'),
 Timestamp('2001-02-26 00:00:00'),
 Timestamp('2001-02-27 00:00:00'),
 Timestamp('2001-04-13 00:00:00'),
 Timestamp('2001-04-21 00:00:00'),
 Timestamp('2001-05-01 00:00:00'),
 Timestamp('2001-06-14 00:00:00'),
 Timestamp('2001-09-07 00:00:00'),
 Timestamp('2001-10-12 00:00:00'),
 Timestamp('2001-11-02 00:00:00'),
 Timestamp('2001-11-15 00:00:00'),
 Timestamp('2001-12-25 00:00:00'),
 Timestamp('2002-01-01 00:00:00'),
 Timestamp('2002-02-11 00:00:00'),
 Timestamp('2002-02-12 00:00:00'),
 Timestamp('2002-03-29 00:00:00'),
 Timestamp('2002-04-21 00:00:00'),
 Timestamp('2002-05-01 00:00:00'),
 Timestamp('2002-05-30 00:00:00'),
 Timestamp('2002-09-07 00:00:00'),
 Timestamp('2002-10-12 00:00:00'),
 Timestamp('2002-11-02 00:00:00'),
 Timestamp('2002-11-15 00:00:00'),
 Timestamp('2002-12-25 00:00:00'),
 Timestamp('2003-01-01 00:00:00'),
 Timestamp('2003-03-03 00:00:00'),
 Timestamp('2003-03-04 00:00:00'),
 Timestamp('2003-04-18 00:00:00'),
 Timestamp('2003-04-

## 1. Criando títulos individuais

Abaixo criamos exemplos simples de cada tipo de título, com parâmetros fixos. Estes exemplos são equivalentes ao que a API faz internamente nos endpoints:
- `POST /titulos/ltn`
- `POST /titulos/lft`
- `POST /titulos/ntnb`
- `POST /titulos/ntnf`

In [2]:
# Datas de exemplo (ajuste se necessário)
data_base = datetime.now().strftime("%Y-%m-%d")

# Exemplo LTN: taxa prefixada
ltn = LTN(
    data_vencimento_titulo="2027-01-01",
    data_base=data_base,
    dias_liquidacao=1,
    taxa=12.50,  # % a.a.
)
ltn.quantidade = 50_000

# Exemplo LFT: pós-fixado ao DI
lft = LFT(
    data_vencimento_titulo="2026-03-01",
    data_base=data_base,
    dias_liquidacao=1,
    taxa=12.00,  # pode ser usado como taxa alvo
)
lft.financeiro = 100_000

# Exemplo NTNB: indexado ao IPCA
ntnb = NTNB(
    data_vencimento_titulo="2035-05-15",
    data_base=data_base,
    dias_liquidacao=1,
    taxa=7.50,  # % a.a. real
)
ntnb.financeiro = 200_000

# Exemplo NTNF: prefixado com prêmio sobre DI
ntnf = NTNF(
    data_vencimento_titulo="2031-01-01",
    data_base=data_base,
    dias_liquidacao=1,
    taxa=11.00,  # % a.a.
)
ntnf.quantidade = 30_000

[OK] Usando cache existente de ANBIMAS completo.


ValueError: Vencimento 2027-01-01 não encontrado na ANBIMA.

## 2. Resumo dos resultados principais

Aqui organizamos os principais resultados de cada título em um `DataFrame` para visualização rápida.

Os campos mostrados incluem:
- `pu_d0`: preço unitário à vista
- `pu_termo` / `pu_carregado` (quando disponíveis)
- `taxa`, `dv01`, `carrego_brl`, `carrego_bps`
- `financeiro` / `quantidade`

In [ ]:
def resumo_titulo(nome_exibicao, titulo):
    return {
        "tipo": nome_exibicao,
        "data_vencimento": getattr(titulo, "_data_vencimento_titulo", None),
        "data_base": getattr(titulo, "data_base", None),
        "dias_liquidacao": getattr(titulo, "dias_liquidacao", None),
        "taxa": getattr(titulo, "taxa", None),
        "quantidade": getattr(titulo, "quantidade", None),
        "financeiro": getattr(titulo, "financeiro", None),
        "pu_d0": getattr(titulo, "pu_d0", None),
        "pu_termo": getattr(titulo, "pu_termo", None),
        "pu_carregado": getattr(titulo, "pu_carregado", None),
        "dv01": getattr(titulo, "dv01", None),
        "carrego_brl": getattr(titulo, "carrego_brl", None),
        "carrego_bps": getattr(titulo, "carrego_bps", None),
    }

titulos = [
    resumo_titulo("LTN", ltn),
    resumo_titulo("LFT", lft),
    resumo_titulo("NTNB", ntnb),
    resumo_titulo("NTNF", ntnf),
]

df_resultados = pd.DataFrame(titulos)
df_resultados

## 3. Exemplo de equivalência entre títulos

Por fim, usamos a função de **equivalência** do módulo de domínio para calcular a quantidade equivalente de um título em relação a outro, com base em um critério (DV01 ou financeiro).

Este fluxo é equivalente ao endpoint:
- `POST /equivalencia`

In [ ]:
# Exemplo: equivalência de LTN -> NTNB usando critério de DV01
qtd_ltn = 10_000

equiv_ntnb = equivalencia(
    titulo1="LTN",
    venc1="2027-01-01",
    titulo2="NTNB",
    venc2="2035-05-15",
    qtd1=qtd_ltn,
    criterio="dv",  # "dv" (DV01) ou "fin" (financeiro)
)

print(f"Quantidade de NTNB equivalente a {qtd_ltn:,.0f} LTN (critério DV01): {equiv_ntnb:,.4f}")

## 4. Próximos passos

- Você pode alterar os parâmetros (datas, taxas, quantidades) para testar diferentes cenários.
- Para validar o funcionamento integrado com a API e o Dash:
  - Inicie a API com `python run_api.py`.
  - Inicie o Dash com `python run_dash_app.py`.
  - Compare os resultados exibidos aqui com os resultados da interface.

Este notebook serve como **teste rápido da calculadora de domínio (`titulospub`)** e como documentação viva de como instanciar e utilizar os títulos diretamente em Python.